# Simulation parameters

In [2]:
import numpy as np
import pandas as pd

cs = 1 / 3**0.5

params_df = pd.DataFrame()
target_dp = np.array([0.1, 0.25, 0.5, 1, 2])
target_Re = np.array(
    [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10]
)
params_df["Case"] = np.arange(0, len(target_dp) * len(target_Re), 1)
params_df["Re"] = np.repeat(target_Re, len(target_dp))
params_df["dp"] = np.tile(target_dp, len(target_Re))
params_df["u"] = cs * 0.1
params_df["viscosity"] = params_df["u"] * params_df["dp"] / params_df["Re"]
params_df["tau"] = 0.5 + params_df["viscosity"] / cs**2

mapping = params_df["tau"] > 1
params_df.loc[mapping, "viscosity"] = (0.9 - 0.5) * cs**2
params_df.loc[mapping, "u"] = (
    params_df.loc[mapping, "Re"]
    * params_df.loc[mapping, "viscosity"]
    / params_df.loc[mapping, "dp"]
)
params_df.loc[mapping, "tau"] = 0.9
params_df.to_csv("./post/006_fixedSphere_PIBM/simulation.csv", index=False)

# Templates

In [3]:
from jinja2 import Template
from pathlib import Path
import pandas as pd
import shutil

folder_path = Path("./post/006_fixedSphere_PIBM")
template_path = folder_path / "templates"
rendered_path = folder_path / "rendered"
full_params = pd.read_csv(folder_path / "simulation.csv")

constants_template: Template = Template((template_path / "constants.j2").read_text())
output_template: Template = Template((template_path / "output.j2").read_text())
particles_template: Template = Template(
    (template_path / "particleCreation.j2").read_text()
)


for idx, case_params in full_params.iterrows():
    target_folder = rendered_path / (
        "006_fixedSphere_PIBM_" + str(int(case_params["Case"]))
    )
    if not target_folder.exists():
        target_folder.mkdir(parents=True)
    (target_folder / "constants.inc").write_text(
        constants_template.render(
            REYNOLDS_NUMBER=case_params["Re"],
            INLET_VELOCITY=case_params["u"],
            PARTICLE_DIAMETER=case_params["dp"],
        )
    )
    (target_folder / "output.inc").write_text(
        output_template.render(
            CASE_LABEL="Case_" + str(int(case_params["Case"])),
        )
    )
    (target_folder / "particleCreation.inc").write_text(
        particles_template.render(
            PARTICLE_DIAMETER=case_params["dp"],
        )
    )
    shutil.copy(
        (template_path / "bc_definition.inc"), (target_folder / "bc_definition.inc")
    )
    shutil.copy(
        (template_path / "bc_initialization.inc"),
        (target_folder / "bc_initialization.inc"),
    )
    shutil.copy((template_path / "model.inc"), (target_folder / "model.inc"))
    shutil.copy(
        (template_path / "flow_initialization.inc"),
        (target_folder / "flow_initialization.inc"),
    )

full_params

,Case,Re,dp,u,viscosity,tau
0,0,0.001,0.10,0.001333,0.133333,0.900000
1,1,0.001,0.25,0.000533,0.133333,0.900000
2,2,0.001,0.50,0.000267,0.133333,0.900000
3,3,0.001,1.00,0.000133,0.133333,0.900000
4,4,0.001,2.00,0.000067,0.133333,0.900000
...,...,...,...,...,...,...
60,60,10.000,0.10,0.057735,0.000577,0.501732
61,61,10.000,0.25,0.057735,0.001443,0.504330
62,62,10.000,0.50,0.057735,0.002887,0.508660
63,63,10.000,1.00,0.057735,0.005774,0.517321


# Simulation

In [4]:
from pathlib import Path
import pandas as pd
import shutil
import subprocess

for idx, case_params in full_params.iterrows():
    # for idx, case_params in [(i, c) for i, c in full_params.iterrows()][:2]:
    var_template: Template = Template((template_path / "var.j2").read_text())
    Path("./src/var.h").write_text(
        var_template.render(
            CASE_FOLDER=f"{int(case_params['Case'])}",
        )
    )
    print(f"Compiling case {int(case_params['Case'])}")
    # with subprocess.Popen(
    #     [
    #         r"A:\Programas\Git\usr\bin\sh.exe",
    #         "--login",
    #         "-c",
    #         f"sh compile.sh D3Q27 000",
    #     ],
    #     cwd=Path("./src").absolute(),
    #     stdout=subprocess.PIPE,
    #     stderr=subprocess.STDOUT,
    #     shell=True,
    #     text=True,
    #     bufsize=1,
    # ) as proc:
    #     for line in proc.stdout:
    #         print(line, end="")
    subprocess.run(
        [
            r"A:\Programas\Git\usr\bin\sh.exe",
            "--login",
            "-c",
            f"sh compile.sh D3Q27 000",
        ],
        cwd=Path("./src").absolute(),
        shell=True,
        check=True,
        capture_output=True,
        text=True,
    )
    print(f"Compiled case {int(case_params['Case'])}")
    print(f"Running case {int(case_params['Case'])}")
    executable_dir = Path("./bin")
    executable_path = executable_dir / "000sim_D3Q27_sm75.exe"
    # with subprocess.Popen(
    #     [executable_path.absolute()],
    #     cwd=executable_dir,
    #     stdout=subprocess.PIPE,
    #     stderr=subprocess.STDOUT,
    #     text=True,
    #     bufsize=1,
    # ) as proc:
    #     for line in proc.stdout:
    #         print(line, end="")
    subprocess.run(
        [executable_path.absolute()],
        cwd=executable_dir,
        check=True,
        capture_output=True,
        text=True,
    )
    print(f"Ran case {int(case_params['Case'])}")

Compiling case 0
Compiled case 0
Running case 0
Ran case 0
Compiling case 1
Compiled case 1
Running case 1
Ran case 1
Compiling case 2
Compiled case 2
Running case 2
Ran case 2
Compiling case 3
Compiled case 3
Running case 3
Ran case 3
Compiling case 4
Compiled case 4
Running case 4
Ran case 4
Compiling case 5
Compiled case 5
Running case 5
Ran case 5
Compiling case 6
Compiled case 6
Running case 6
Ran case 6
Compiling case 7
Compiled case 7
Running case 7
Ran case 7
Compiling case 8
Compiled case 8
Running case 8
Ran case 8
Compiling case 9
Compiled case 9
Running case 9
Ran case 9
Compiling case 10
Compiled case 10
Running case 10
Ran case 10
Compiling case 11
Compiled case 11
Running case 11
Ran case 11
Compiling case 12
Compiled case 12
Running case 12
Ran case 12
Compiling case 13
Compiled case 13
Running case 13
Ran case 13
Compiling case 14
Compiled case 14
Running case 14
Ran case 14
Compiling case 15
Compiled case 15
Running case 15
Ran case 15
Compiling case 16
Compiled case 